# RQ1 Fair Comparison + Hybrid Sweep

This notebook implements:
1. Fair comparison with matched protocol for V2 and V3 under scalar threshold only.
2. Fair comparison with matched protocol for V2 and V3 under per-label threshold only.
3. Hybrid sweep at fixed epsilon over lr, max_grad_norm, epochs, warmup_ratio with weight_decay fixed at 0.01.
4. Hybrid calibration after training using temperature scaling before per-label threshold search.
5. Multi-seed stability reporting (mean and std).

In [1]:
!pip install opacus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.4/254.4 kB 6.7 MB/s eta 0:00:00


In [ ]:
import os
import sys
import copy
import random
import importlib.util
from dataclasses import dataclass
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

from opacus.validators import ModuleValidator

REPO_NAME = 'Hybrid-DP-Approach-For-Mental-Health-Chatbots'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def _looks_like_repo(root: Path) -> bool:
    return (root / 'src' / 'utils' / 'hybrid_dp_utils.py').exists() and (root / 'data').exists()


def _find_repo_by_name(base: Path, repo_name: str, max_depth: int = 5):
    if not base.exists():
        return None
    base = base.resolve()
    for current, dirs, _ in os.walk(base):
        current_path = Path(current)
        depth = len(current_path.relative_to(base).parts)
        if depth > max_depth:
            dirs[:] = []
            continue

        if current_path.name == repo_name and _looks_like_repo(current_path):
            return current_path

    return None


def resolve_project_root() -> Path:
    cwd = Path.cwd()
    probe = cwd
    while probe.parent != probe:
        if _looks_like_repo(probe):
            return probe
        probe = probe.parent

    
    if IN_COLAB:
        exact_candidates = [
            Path('/content/drive/MyDrive') / REPO_NAME,
            Path('/content/drive/My Drive') / REPO_NAME,
        ]
        for c in exact_candidates:
            if _looks_like_repo(c):
                return c

        
        scan_bases = [
            Path('/content/drive/MyDrive'),
            Path('/content/drive/My Drive'),
            Path('/content'),
        ]
        for base in scan_bases:
            found = _find_repo_by_name(base, REPO_NAME, max_depth=6)
            if found is not None:
                return found

    return cwd


PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from src.utils.hybrid_dp_utils import HybridDPCoach
except ModuleNotFoundError:
    util_path = PROJECT_ROOT / 'src' / 'utils' / 'hybrid_dp_utils.py'
    if not util_path.exists():
        raise ModuleNotFoundError(
            f"Could not import HybridDPCoach. Resolved PROJECT_ROOT={PROJECT_ROOT}, "
            f"missing {util_path}"
        )

    spec = importlib.util.spec_from_file_location('hybrid_dp_utils', util_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    HybridDPCoach = module.HybridDPCoach

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {device}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/content/Hybrid-DP-Approach-For-Mental-Health-Chatbots
Device: cuda


In [ ]:
MODEL_NAME = 'roberta-base'
MAX_LEN = 128
BATCH_SIZE = 16
FIXED_EPSILON = 8.0
DELTA = 1e-5
SEEDS = [7, 13, 23]

FAIR_SCALAR_GRID = np.round(np.arange(0.05, 0.60, 0.05), 2).tolist()
FAIR_PER_LABEL_GRID = np.round(np.arange(0.05, 0.60, 0.05), 2).tolist()

SWEEP_LR = [1e-5, 2e-5, 3e-5]
SWEEP_MAX_GRAD_NORM = [0.6, 0.8, 1.0, 1.2]
SWEEP_EPOCHS = [6, 8, 10, 12]
SWEEP_WARMUP = [0.06, 0.10]
SWEEP_WEIGHT_DECAY = 0.01

RUN_FULL_HYBRID_SWEEP = True

DATALOADER_NUM_WORKERS = min(4, os.cpu_count() or 1)
DATALOADER_PIN_MEMORY = torch.cuda.is_available()

TOKENIZER = None
def get_tokenizer():
    global TOKENIZER
    if TOKENIZER is None:
        TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    return TOKENIZER

REPORT_DIR = PROJECT_ROOT / 'reports' / 'rq1'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

EMOTION_COLS = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral',
]

RAW_CANDIDATES = [
    PROJECT_ROOT / 'data' / 'raw' / 'go_emotions_dataset.csv',
    PROJECT_ROOT / 'content' / 'data' / 'raw' / 'go_emotions_dataset.csv',
]
CLEANED_CANDIDATES = [
    PROJECT_ROOT / 'data' / 'processed' / 'go_emotions_cleaned.csv',
    PROJECT_ROOT / 'content' / 'data' / 'processed' / 'go_emotions_cleaned.csv',
]

RAW_DATA_PATH = next((p for p in RAW_CANDIDATES if p.exists()), RAW_CANDIDATES[0])
CLEANED_DATA_PATH = next((p for p in CLEANED_CANDIDATES if p.exists()), CLEANED_CANDIDATES[0])

assert RAW_DATA_PATH.exists(), f'Missing file: {RAW_DATA_PATH}'
assert CLEANED_DATA_PATH.exists(), f'Missing file: {CLEANED_DATA_PATH}'
print(f'Raw path selected: {RAW_DATA_PATH}')
print(f'Cleaned path selected: {CLEANED_DATA_PATH}')
print('Config ready.')

Raw path selected: /content/drive/MyDrive/content/Hybrid-DP-Approach-For-Mental-Health-Chatbots/data/raw/go_emotions_dataset.csv
Cleaned path selected: /content/drive/MyDrive/content/Hybrid-DP-Approach-For-Mental-Health-Chatbots/data/processed/go_emotions_cleaned.csv
Config ready.


In [4]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        # Pre-tokenize once to avoid repeating tokenization every epoch.
        texts = [str(t) for t in texts]
        enc = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_len,
            return_tensors='pt'
        )
        self.input_ids = enc['input_ids']
        self.attention_mask = enc['attention_mask']
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }


def load_aligned_variants():
    df_raw = pd.read_csv(RAW_DATA_PATH)
    df_cleaned = pd.read_csv(CLEANED_DATA_PATH)

    meta_cols = {'id', 'text', 'example_very_unclear', 'clean_text', 'num_emotions', 'emotion_bin'}
    emotion_cols = [c for c in df_raw.columns if c not in meta_cols]

    required_clean_cols = {'id', 'clean_text'}
    missing_clean = required_clean_cols - set(df_cleaned.columns)
    if missing_clean:
        raise ValueError(f'Missing required cleaned columns: {sorted(missing_clean)}')

    merged = df_raw[['id', 'text'] + emotion_cols].merge(
        df_cleaned[['id', 'clean_text']],
        on='id',
        how='inner'
    )

    merged['text_v0'] = merged['text'].astype(str).str.lower()
    merged['text_v1'] = merged['clean_text'].astype(str).str.lower()
    merged['labels'] = merged[emotion_cols].values.tolist()

    global EMOTION_COLS
    EMOTION_COLS = emotion_cols
    print(f'Loaded RQ1-aligned data with {len(EMOTION_COLS)} labels.')

    return merged[['id', 'text_v0', 'text_v1'] + EMOTION_COLS + ['labels']]


def split_ids(df, seed=42):
    strat = df[EMOTION_COLS].sum(axis=1)
    bins = pd.qcut(strat, q=3, duplicates='drop')

    idx = np.arange(len(df))
    train_idx, temp_idx = train_test_split(
        idx,
        test_size=0.4,
        random_state=seed,
        stratify=bins
    )
    temp_bins = bins.iloc[temp_idx]
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=0.5,
        random_state=seed,
        stratify=temp_bins
    )

    train_ids = df.iloc[train_idx]['id'].tolist()
    val_ids = df.iloc[val_idx]['id'].tolist()
    test_ids = df.iloc[test_idx]['id'].tolist()
    return set(train_ids), set(val_ids), set(test_ids)


def pick_variant_text(df, variant):
    if variant == 'V2_DP_SGD':
        return df['text_v0'].tolist()
    if variant == 'V3_HYBRID':
        return df['text_v1'].tolist()
    raise ValueError(f'Unsupported variant: {variant}')


full_df = load_aligned_variants()
train_ids, val_ids, test_ids = split_ids(full_df, seed=42)

df_train = full_df[full_df['id'].isin(train_ids)].reset_index(drop=True)
df_val = full_df[full_df['id'].isin(val_ids)].reset_index(drop=True)
df_test = full_df[full_df['id'].isin(test_ids)].reset_index(drop=True)

print(f'Train={len(df_train)} Val={len(df_val)} Test={len(df_test)}')

Loaded RQ1-aligned data with 28 labels.
Train=200657 Val=119113 Test=119362


In [5]:
def logits_to_probs(logits: np.ndarray):
    return 1.0 / (1.0 + np.exp(-logits))


def evaluate_with_thresholds(logits: np.ndarray, y_true: np.ndarray, thresholds):
    probs = logits_to_probs(logits)
    if np.isscalar(thresholds):
        preds = (probs >= float(thresholds)).astype(int)
    else:
        thr = np.asarray(thresholds, dtype=float).reshape(1, -1)
        preds = (probs >= thr).astype(int)

    return {
        'f1_macro': f1_score(y_true, preds, average='macro', zero_division=0),
        'f1_micro': f1_score(y_true, preds, average='micro', zero_division=0),
        'precision_micro': precision_score(y_true, preds, average='micro', zero_division=0),
        'recall_micro': recall_score(y_true, preds, average='micro', zero_division=0),
    }


def fit_temperature(logits: np.ndarray, y_true: np.ndarray, max_iter=200):
    logits_t = torch.tensor(logits, dtype=torch.float32)
    labels_t = torch.tensor(y_true, dtype=torch.float32)

    log_temp = torch.nn.Parameter(torch.tensor(0.0))
    optimizer = torch.optim.LBFGS([log_temp], lr=0.05, max_iter=max_iter)
    criterion = torch.nn.BCEWithLogitsLoss()

    def closure():
        optimizer.zero_grad()
        temp = torch.exp(log_temp) + 1e-6
        loss = criterion(logits_t / temp, labels_t)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float((torch.exp(log_temp) + 1e-6).detach().cpu().item())


def best_scalar_threshold(logits: np.ndarray, y_true: np.ndarray, grid):
    best_thr, best_f1 = 0.5, -1.0
    for thr in grid:
        f1 = evaluate_with_thresholds(logits, y_true, thr)['f1_macro']
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr
    return float(best_thr), float(best_f1)


def best_per_label_thresholds(logits: np.ndarray, y_true: np.ndarray, grid):
    probs = logits_to_probs(logits)
    n_labels = y_true.shape[1]
    thr_vec = np.zeros(n_labels, dtype=float)

    for j in range(n_labels):
        yj = y_true[:, j]
        best_thr, best_f1 = 0.5, -1.0
        for thr in grid:
            pj = (probs[:, j] >= thr).astype(int)
            f1j = f1_score(yj, pj, zero_division=0)
            if f1j > best_f1:
                best_f1 = f1j
                best_thr = thr
        thr_vec[j] = best_thr

    macro_f1 = evaluate_with_thresholds(logits, y_true, thr_vec)['f1_macro']
    return thr_vec, float(macro_f1)


def collect_logits(model, data_loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device, non_blocking=DATALOADER_PIN_MEMORY)
            attention_mask = batch['attention_mask'].to(device, non_blocking=DATALOADER_PIN_MEMORY)
            labels = batch['labels'].cpu().numpy()

            out = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = out.logits.detach().cpu().numpy()
            all_logits.append(logits)
            all_labels.append(labels)

    return np.vstack(all_logits), np.vstack(all_labels)

In [ ]:
@dataclass
class TrainConfig:
    variant: str
    seed: int
    epsilon: float
    use_dp: bool
    lr: float
    max_grad_norm: float
    epochs: int
    warmup_ratio: float
    weight_decay: float
    threshold_mode: str
    use_temperature_scaling: bool


def train_and_eval(cfg: TrainConfig):
    set_seed(cfg.seed)

    tokenizer = get_tokenizer()
    n_labels = len(EMOTION_COLS)

    train_texts = pick_variant_text(df_train, cfg.variant)
    val_texts = pick_variant_text(df_val, cfg.variant)
    test_texts = pick_variant_text(df_test, cfg.variant)

    y_train = np.asarray(df_train['labels'].tolist(), dtype=np.float32)
    y_val = np.asarray(df_val['labels'].tolist(), dtype=np.float32)
    y_test = np.asarray(df_test['labels'].tolist(), dtype=np.float32)

    train_ds = EmotionDataset(train_texts, y_train, tokenizer, max_len=MAX_LEN)
    val_ds = EmotionDataset(val_texts, y_val, tokenizer, max_len=MAX_LEN)
    test_ds = EmotionDataset(test_texts, y_test, tokenizer, max_len=MAX_LEN)

    loader_kwargs = {
        'batch_size': BATCH_SIZE,
        'num_workers': DATALOADER_NUM_WORKERS,
        'pin_memory': DATALOADER_PIN_MEMORY
    }
    if DATALOADER_NUM_WORKERS > 0:
        loader_kwargs['persistent_workers'] = True

    train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=n_labels,
        problem_type='multi_label_classification'
    )

    if cfg.use_dp:
        model.train()
        model = ModuleValidator.fix(model)
        model.train()

    model = model.to(device)
    if cfg.use_dp:
        model.train()

    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay
    )

    if cfg.use_dp:
        coach = HybridDPCoach(
            model=model,
            optimizer=optimizer,
            data_loader=train_loader,
            epsilon=cfg.epsilon,
            delta=DELTA,
            max_grad_norm=cfg.max_grad_norm,
            epochs=cfg.epochs
        )
        model, optimizer, train_loader = coach.attach()
        model.train()
    else:
        coach = None

    total_steps = len(train_loader) * cfg.epochs
    warmup_steps = int(cfg.warmup_ratio * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    best_state = None
    best_val_loss = float('inf')

    epoch_bar = tqdm(range(cfg.epochs), desc=f'Train {cfg.variant} seed={cfg.seed}', leave=True)
    for epoch_idx in epoch_bar:
        model.train()
        running_train_loss = 0.0
        batch_bar = tqdm(
            train_loader,
            total=len(train_loader),
            desc=f'Epoch {epoch_idx + 1}/{cfg.epochs}',
            leave=False
        )

        for step_idx, batch in enumerate(batch_bar, start=1):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device, non_blocking=DATALOADER_PIN_MEMORY)
            attention_mask = batch['attention_mask'].to(device, non_blocking=DATALOADER_PIN_MEMORY)
            labels = batch['labels'].to(device, non_blocking=DATALOADER_PIN_MEMORY)

            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = out.loss
            loss.backward()
            optimizer.step()
            scheduler.step()

            running_train_loss += loss.item()
            if step_idx % max(1, len(train_loader) // 20) == 0 or step_idx == len(train_loader):
                batch_bar.set_postfix(
                    train_loss=f'{running_train_loss / step_idx:.4f}',
                    lr=f'{scheduler.get_last_lr()[0]:.2e}'
                )

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device, non_blocking=DATALOADER_PIN_MEMORY)
                attention_mask = batch['attention_mask'].to(device, non_blocking=DATALOADER_PIN_MEMORY)
                labels = batch['labels'].to(device, non_blocking=DATALOADER_PIN_MEMORY)
                out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                val_losses.append(out.loss.item())

        mean_train_loss = float(running_train_loss / max(1, len(train_loader)))
        mean_val_loss = float(np.mean(val_losses))
        epoch_bar.set_postfix(train_loss=f'{mean_train_loss:.4f}', val_loss=f'{mean_val_loss:.4f}')
        print(f'Epoch {epoch_idx + 1}/{cfg.epochs} | train_loss={mean_train_loss:.4f} | val_loss={mean_val_loss:.4f}')

        if mean_val_loss < best_val_loss:
            best_val_loss = mean_val_loss
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    val_logits, y_val_true = collect_logits(model, val_loader)
    test_logits, y_test_true = collect_logits(model, test_loader)

    temperature = 1.0
    if cfg.use_temperature_scaling:
        temperature = fit_temperature(val_logits, y_val_true)
        val_logits = val_logits / temperature
        test_logits = test_logits / temperature

    threshold_used = None
    if cfg.threshold_mode == 'scalar':
        threshold_used, _ = best_scalar_threshold(val_logits, y_val_true, FAIR_SCALAR_GRID)
    elif cfg.threshold_mode == 'per_label':
        threshold_used, _ = best_per_label_thresholds(val_logits, y_val_true, FAIR_PER_LABEL_GRID)
    else:
        raise ValueError(f'Unknown threshold_mode: {cfg.threshold_mode}')

    val_metrics = evaluate_with_thresholds(val_logits, y_val_true, threshold_used)
    test_metrics = evaluate_with_thresholds(test_logits, y_test_true, threshold_used)

    result = {
        'variant': cfg.variant,
        'seed': cfg.seed,
        'epsilon': cfg.epsilon,
        'use_dp': cfg.use_dp,
        'lr': cfg.lr,
        'max_grad_norm': cfg.max_grad_norm,
        'epochs': cfg.epochs,
        'warmup_ratio': cfg.warmup_ratio,
        'weight_decay': cfg.weight_decay,
        'threshold_mode': cfg.threshold_mode,
        'temperature_scaling': cfg.use_temperature_scaling,
        'temperature': temperature,
        'threshold_used': threshold_used.tolist() if isinstance(threshold_used, np.ndarray) else threshold_used,
        'val_f1_macro': val_metrics['f1_macro'],
        'val_f1_micro': val_metrics['f1_micro'],
        'test_f1_macro': test_metrics['f1_macro'],
        'test_f1_micro': test_metrics['f1_micro'],
        'test_precision_micro': test_metrics['precision_micro'],
        'test_recall_micro': test_metrics['recall_micro'],
        'privacy_spent_epsilon': coach.get_privacy_spent() if coach is not None else 0.0,
    }
    return result

In [7]:
def summarize_mean_std(df, group_cols, metrics):
    out = df.groupby(group_cols, dropna=False)[metrics].agg(['mean', 'std']).reset_index()
    out.columns = [
        '_'.join([c for c in col if c]).rstrip('_') if isinstance(col, tuple) else col
        for col in out.columns.values
    ]
    return out


def _load_existing_or_empty(path):
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()


def _append_run_row(path, row_dict):
    row_df = pd.DataFrame([row_dict])
    if path.exists():
        prev = pd.read_csv(path)
        row_df = pd.concat([prev, row_df], ignore_index=True)
    row_df.to_csv(path, index=False)


def run_fair_comparison(resume=True):
    runs_path = REPORT_DIR / 'rq1_fair_v2_v3_runs.csv'
    summary_path = REPORT_DIR / 'rq1_fair_v2_v3_summary.csv'

    existing = _load_existing_or_empty(runs_path) if resume else pd.DataFrame()
    completed = set()
    if not existing.empty:
        for _, r in existing.iterrows():
            completed.add((str(r['threshold_mode']), int(r['seed']), str(r['variant'])))

    for threshold_mode in ['scalar', 'per_label']:
        for seed in SEEDS:
            for variant in ['V2_DP_SGD', 'V3_HYBRID']:
                key = (threshold_mode, int(seed), variant)
                if key in completed:
                    print(f'Skipping completed run: mode={threshold_mode}, seed={seed}, variant={variant}')
                    continue

                cfg = TrainConfig(
                    variant=variant,
                    seed=seed,
                    epsilon=FIXED_EPSILON,
                    use_dp=True,
                    lr=2e-5,
                    max_grad_norm=1.0,
                    epochs=8,
                    warmup_ratio=0.06,
                    weight_decay=SWEEP_WEIGHT_DECAY,
                    threshold_mode=threshold_mode,
                    use_temperature_scaling=False
                )

                row = train_and_eval(cfg)
                _append_run_row(runs_path, row)
                print(f'Checkpoint saved: {runs_path} | mode={threshold_mode}, seed={seed}, variant={variant}')

    runs = _load_existing_or_empty(runs_path)
    if not runs.empty:
        summary = summarize_mean_std(
            runs,
            group_cols=['threshold_mode', 'variant'],
            metrics=['test_f1_macro', 'test_f1_micro', 'test_precision_micro', 'test_recall_micro']
        )
        summary.to_csv(summary_path, index=False)
    else:
        summary = pd.DataFrame()

    return runs, summary


def run_hybrid_sweep(run_full=True, resume=True):
    runs_path = REPORT_DIR / 'rq1_hybrid_sweep_runs.csv'
    summary_path = REPORT_DIR / 'rq1_hybrid_sweep_summary.csv'
    best_path = REPORT_DIR / 'rq1_hybrid_sweep_best_config.csv'

    existing = _load_existing_or_empty(runs_path) if resume else pd.DataFrame()
    completed = set()
    if not existing.empty:
        for _, r in existing.iterrows():
            completed.add((
                float(r['lr']),
                float(r['max_grad_norm']),
                int(r['epochs']),
                float(r['warmup_ratio']),
                float(r['weight_decay']),
                int(r['seed'])
            ))

    all_cfgs = list(product(SWEEP_LR, SWEEP_MAX_GRAD_NORM, SWEEP_EPOCHS, SWEEP_WARMUP))
    if not run_full:
        all_cfgs = all_cfgs[:8]

    for lr, max_grad_norm, epochs, warmup in all_cfgs:
        for seed in SEEDS:
            key = (float(lr), float(max_grad_norm), int(epochs), float(warmup), float(SWEEP_WEIGHT_DECAY), int(seed))
            if key in completed:
                print(f'Skipping completed sweep run: lr={lr}, clip={max_grad_norm}, epochs={epochs}, warmup={warmup}, seed={seed}')
                continue

            cfg = TrainConfig(
                variant='V3_HYBRID',
                seed=seed,
                epsilon=FIXED_EPSILON,
                use_dp=True,
                lr=lr,
                max_grad_norm=max_grad_norm,
                epochs=epochs,
                warmup_ratio=warmup,
                weight_decay=SWEEP_WEIGHT_DECAY,
                threshold_mode='per_label',
                use_temperature_scaling=True
            )

            row = train_and_eval(cfg)
            _append_run_row(runs_path, row)
            print(f'Checkpoint saved: {runs_path} | lr={lr}, clip={max_grad_norm}, epochs={epochs}, warmup={warmup}, seed={seed}')

    runs = _load_existing_or_empty(runs_path)
    if not runs.empty:
        summary = summarize_mean_std(
            runs,
            group_cols=['lr', 'max_grad_norm', 'epochs', 'warmup_ratio', 'weight_decay'],
            metrics=['test_f1_macro', 'test_f1_micro', 'test_precision_micro', 'test_recall_micro', 'temperature']
        )
        summary = summary.sort_values('test_f1_macro_mean', ascending=False).reset_index(drop=True)
        summary.to_csv(summary_path, index=False)

        if len(summary) > 0:
            best_row = summary.iloc[0].to_dict()
            pd.DataFrame([best_row]).to_csv(best_path, index=False)
    else:
        summary = pd.DataFrame()

    return runs, summary

In [8]:
# Execute fair V2 vs V3 comparisons
fair_runs, fair_summary = run_fair_comparison()
print('Saved:', REPORT_DIR / 'rq1_fair_v2_v3_runs.csv')
print('Saved:', REPORT_DIR / 'rq1_fair_v2_v3_summary.csv')
display(fair_summary)

Skipping completed run: mode=scalar, seed=7, variant=V2_DP_SGD
Skipping completed run: mode=scalar, seed=7, variant=V3_HYBRID
Skipping completed run: mode=scalar, seed=13, variant=V2_DP_SGD
Skipping completed run: mode=scalar, seed=13, variant=V3_HYBRID
Skipping completed run: mode=scalar, seed=23, variant=V2_DP_SGD


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is

Train V2_DP_SGD seed=7:   0%|          | 0/8 [00:00<?, ?it/s]

Epoch 1/8:   0%|          | 0/12542 [00:00<?, ?it/s]

sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


KeyboardInterrupt: 

In [ ]:
# Execute Hybrid sweep with temperature scaling + per-label threshold calibration
sweep_runs, sweep_summary = run_hybrid_sweep(run_full=RUN_FULL_HYBRID_SWEEP)
print('Saved:', REPORT_DIR / 'rq1_hybrid_sweep_runs.csv')
print('Saved:', REPORT_DIR / 'rq1_hybrid_sweep_summary.csv')
print('Saved:', REPORT_DIR / 'rq1_hybrid_sweep_best_config.csv')
display(sweep_summary.head(10))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is

Train V3_HYBRID seed=7:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch 1/6:   0%|          | 0/12542 [00:00<?, ?it/s]

sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
